# Spatial Protein Clustering Analysis

This notebook identifies distinct spatial compartments in cells by clustering
pixels based on their multi-protein expression profiles using ProtVS predictions.

**Workflow:**
1. Generate protein localization predictions with ProtVS
2. Save predictions to disk (memory-efficient)
3. Load predictions and cluster pixels with KMeans
4. Assign genes to clusters by specificity
5. Visualize spatial compartment maps

## Setup

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
from collections import defaultdict
from pathlib import Path

# from skimage.filters import threshold_otsu, gaussian
from sklearn.cluster import KMeans
from tifffile import imread, imwrite
from tqdm.auto import tqdm

from protvs import ProtVS

c:\Users\57539\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [3]:
# Analysis parameters
N_CLUSTERS = 10
CELL_LINE = "A-431"
# 
# Directories
INPUT_IMAGE_DIR = "example_cell_reference_input"
PRED_IMAGE_DIR = "spatial_clustering_output"
RESULT_DIR = "spatial_clustering_results"

os.makedirs(PRED_IMAGE_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

# Color palette for visualization
STANDARD_COLORS = [
    (0, 0, 0), (255, 0, 0), (0, 0, 255), (0, 128, 0), (255, 255, 0),
    (255, 0, 255), (0, 255, 255), (255, 165, 0), (128, 0, 128),
    (0, 128, 128), (135, 206, 235), (128, 0, 0),
]

def rgb_to_hex(rgb):
    return "#{:02x}{:02x}{:02x}".format(int(rgb[0]), int(rgb[1]), int(rgb[2]))

print(f"Clusters: {N_CLUSTERS}")
print(f"Cell line: {CELL_LINE}")

Clusters: 10
Cell line: A-431


## Gene Selection

Select proteins spanning major cellular compartments.

In [4]:
selected_gene_names = [
    # Nuclear envelope
    "TPR", "LMNB1", "SUN2", "TMPO", "LEMD2", "TOR1AIP1", "LBR", "LMNB2", "NUP153"
]

selected_gene_names = list(set(selected_gene_names))
print(f"{len(selected_gene_names)} unique genes selected")

9 unique genes selected


## Step 1: Generate Predictions with ProtVS

Predict each protein for every reference cell image and save to disk.
This step is memory-intensive — predictions are saved as TIFFs so we
can free GPU memory before analysis.

In [5]:
model = ProtVS()

# Collect all reference images
ref_files = sorted([f for f in os.listdir(INPUT_IMAGE_DIR) if f.endswith(".tiff")])
print(f"Found {len(ref_files)} reference images")

# Generate predictions for each cell x gene combination
for cell_file in tqdm(ref_files, desc="Cells"):
    ref_image = imread(os.path.join(INPUT_IMAGE_DIR, cell_file))
    cell_prefix = Path(cell_file).stem

    for gene in tqdm(selected_gene_names, desc=f"  {cell_prefix}", leave=False):
        out_path = os.path.join(PRED_IMAGE_DIR, f"{cell_prefix}_{CELL_LINE}_{gene}_pred.tif")
        if os.path.exists(out_path):
            continue  # skip already generated

        result = model.predict(
            images=[ref_image],
            protein_names=[gene],
            cell_line_names=[CELL_LINE],
            num_inference_steps=50,
        )
        imwrite(out_path, result[0].astype(np.float32))

# Free GPU memory
del model
import torch; torch.cuda.empty_cache()

print(f"Predictions saved to {PRED_IMAGE_DIR}/")

Found 15 reference images


Cells:   0%|          | 0/15 [00:00<?, ?it/s]


c:\Users\57539\AppData\Local\Programs\Python\Python311\Lib\site-packages\diffusers\models\attention_processor.py:3313: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  hidden_states = F.scaled_dot_product_attention(
Cells:   0%|          | 0/15 [00:14<?, ?it/s]


SafetensorError: Error while deserializing header: MetadataIncompleteBuffer

In [ ]:
protvs\checkpoint\unet_ema\diffusion_pytorch_model.safetensors

## Step 2: Preprocessing Helpers

In [ ]:
def preprocess_image_stack(img_stack, thresh_multiplier=1.5, sigma=0.5):
    """Apply Gaussian smoothing and Otsu thresholding to image stack."""
    processed = img_stack.astype(np.float32)
    for i in range(processed.shape[2]):
        processed[:, :, i] = gaussian(processed[:, :, i], sigma=sigma)
        thresh = thresh_multiplier * threshold_otsu(processed[:, :, i])
        processed[:, :, i] = processed[:, :, i] > thresh
    return processed

def calculate_gene_specificity(expression_values, n_clusters):
    """Calculate specificity metrics for a gene across clusters."""
    total_expr = sum(expression_values)
    mean_expr = np.mean(expression_values)
    std_expr = np.std(expression_values)

    metrics = []
    for cluster_idx, expr in enumerate(expression_values):
        if expr == 0:
            continue
        other_expr = total_expr - expr
        mean_other = other_expr / (n_clusters - 1) if n_clusters > 1 else 0
        specificity = expr / (mean_other + 1e-6)
        z_score = (expr - mean_expr) / std_expr if std_expr > 0 else 0
        fold_change = expr / (mean_expr + 1e-6)
        enrichment = (2 * specificity + z_score + fold_change) / 4

        metrics.append({
            "cluster": cluster_idx, "expression": expr,
            "specificity": specificity, "z_score": z_score,
            "enrichment": enrichment, "fraction": expr / total_expr,
        })
    return sorted(metrics, key=lambda x: x["enrichment"], reverse=True)

def assign_gene_to_clusters(metrics, mean_expr, max_clusters=3):
    """Assign gene to clusters based on specificity thresholds."""
    if not metrics:
        return []
    assigned = []
    top = metrics[0]
    if top["z_score"] > 0.5 and top["specificity"] > 1.5 and top["fraction"] > 0.3:
        assigned.append(top["cluster"])
        for m in metrics[1:]:
            if len(assigned) >= max_clusters:
                break
            if (m["enrichment"] > top["enrichment"] * 0.6
                or (m["specificity"] > 1.5 and m["expression"] > mean_expr * 1.5)
                or m["fraction"] > 0.3):
                assigned.append(m["cluster"])
    elif top["expression"] > mean_expr * 0.5:
        assigned.append(top["cluster"])
    return assigned

## Step 3: Load Predictions and Build Pixel Matrix

In [ ]:
import glob

all_files = sorted(glob.glob(os.path.join(PRED_IMAGE_DIR, f"*_{CELL_LINE}_*_pred.tif")))
print(f"Found {len(all_files)} prediction files")

# Stack all gene channels into one image per cell
channels = []
for filepath in tqdm(all_files, desc="Loading"):
    img = imread(filepath).astype(np.float32)
    channels.append(img)

img_stack = np.stack(channels, axis=-1)
img_stack = preprocess_image_stack(img_stack, thresh_multiplier=1.5)

pixels = img_stack.reshape(-1, img_stack.shape[2]).astype(np.int8)
print(f"Pixel matrix: {pixels.shape[0]:,} pixels x {pixels.shape[1]} genes")

## Step 4: KMeans Clustering

In [ ]:
print("Performing KMeans clustering...")
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
kmeans.fit(pixels)

# Sort clusters by size (largest first)
counts = np.bincount(kmeans.labels_)
sorted_indices = np.argsort(-counts)
reordered_centers = kmeans.cluster_centers_[sorted_indices]

print("\nCluster sizes:")
for i, idx in enumerate(sorted_indices):
    print(f"  Cluster {i}: {counts[idx]:,} ({100*counts[idx]/len(kmeans.labels_):.1f}%)")

## Step 5: Gene-Cluster Assignment

In [ ]:
gene_cluster_mapping = {}

for gene_idx, gene_name in enumerate(selected_gene_names):
    expressions = [reordered_centers[c][gene_idx] for c in range(N_CLUSTERS)]
    if max(expressions) == 0:
        continue
    metrics = calculate_gene_specificity(expressions, N_CLUSTERS)
    assigned = assign_gene_to_clusters(metrics, np.mean(expressions))
    if assigned:
        gene_cluster_mapping[gene_name] = {
            "clusters": assigned, "metrics": metrics[:len(assigned)]
        }

print(f"Assigned {len(gene_cluster_mapping)} genes to clusters")

## Step 6: Export Results

In [ ]:
cluster_colors = [rgb_to_hex(STANDARD_COLORS[i]) for i in range(N_CLUSTERS)]

with open(f"{RESULT_DIR}/gene_color_mapping.txt", "w") as f:
    for gene in sorted(gene_cluster_mapping.keys()):
        for cluster_idx in gene_cluster_mapping[gene]["clusters"]:
            f.write(f"{gene} {cluster_colors[cluster_idx]}\n")

with open(f"{RESULT_DIR}/cluster_analysis.txt", "w") as f:
    f.write("CLUSTER ANALYSIS\n" + "=" * 80 + "\n\n")
    for i in range(N_CLUSTERS):
        cluster_genes = []
        for gene, data in gene_cluster_mapping.items():
            if i in data["clusters"]:
                metric = next(m for m in data["metrics"] if m["cluster"] == i)
                cluster_genes.append((gene, metric["specificity"], metric["z_score"]))
        cluster_genes.sort(key=lambda x: x[1], reverse=True)
        f.write(f"Cluster {i} ({cluster_colors[i]})\n")
        f.write(f"  Size: {counts[sorted_indices[i]]:,} pixels\n")
        f.write(f"  Genes: {len(cluster_genes)}\n")
        if cluster_genes:
            f.write(f"  Top genes: {', '.join([g[0] for g in cluster_genes[:5]])}\n")
        f.write("\n")

print(f"Results exported to {RESULT_DIR}/")

## Step 7: Generate Spatial Maps

In [ ]:
print("Generating spatial maps...")

all_files = sorted(glob.glob(os.path.join(PRED_IMAGE_DIR, f"*{CELL_LINE}*_pred.tif")))
cell_groups = defaultdict(list)

for filepath in all_files:
    filename = os.path.basename(filepath).replace("_pred.tif", "")
    for gene in selected_gene_names:
        if gene in filename:
            cell_prefix = filename.replace(f"_{gene}", "").replace(gene, "")
            cell_groups[cell_prefix].append((filepath, gene))
            break

for idx, (cell_prefix, gene_files) in enumerate(sorted(cell_groups.items())):
    gene_files_sorted = sorted(
        gene_files,
        key=lambda x: selected_gene_names.index(x[1]) if x[1] in selected_gene_names else 999,
    )

    channels = []
    for filepath, gene in gene_files_sorted:
        img = imread(filepath).astype(np.float32)
        channels.append(img)

    img_stack = np.stack(channels, axis=-1)
    img_stack = preprocess_image_stack(img_stack, thresh_multiplier=1.25)

    h, w = img_stack.shape[:2]
    pix = img_stack.reshape(-1, img_stack.shape[2]).astype(np.int8)
    labels = kmeans.predict(pix)

    # Re-index to sorted order
    new_labels = np.zeros_like(labels)
    for i, si in enumerate(sorted_indices):
        new_labels[labels == si] = i

    clustered = new_labels.reshape(h, w)
    rgb_img = np.zeros((h, w, 3), dtype=np.uint8)
    for i in range(N_CLUSTERS):
        rgb_img[clustered == i] = STANDARD_COLORS[i]

    safe_prefix = cell_prefix.replace("/", "_")
    plt.imsave(f"{RESULT_DIR}/{safe_prefix}_clustered.png", rgb_img)
    print(f"  Processed {idx + 1} cells", end="\r")

print(f"\nSaved {len(cell_groups)} spatial maps to {RESULT_DIR}/")

## Summary

**Outputs:**
- `gene_color_mapping.txt` — Gene-to-cluster color assignments
- `cluster_analysis.txt` — Cluster sizes, enriched genes
- `*_clustered.png` — Per-cell spatial compartment maps